# About the Notebook

`ch02_harness_components.ipynb` showed how harness components interact over model context, workflow state, operational state, and durable state. This notebook now moves from a component demonstration into a production-shaped state boundary.

The question is:

> **What must the harness own so execution can continue without pretending the model or client remembers?**

The implementation stays small enough to inspect, but the interfaces follow the
boundaries you should preserve in a real system:

1. the UI sends the initial task, but authenticated identity comes from the server;
2. the harness creates a `run_id` and binds it to a tenant and principal;
3. a `run_id` locates a run but never authorizes access to it;
4. workflow state is checkpointed outside model context and process memory;
5. approvals are scoped to a concrete pending action and authenticated approver;
6. external effects use durable idempotency keys;
7. agent handoffs use harness-owned artifacts rather than shared model memory.


## Setup

The notebook uses only the Python standard library and Pydantic. SQLite stands in
for a durable checkpoint store because it gives the example real persistence,
transactions, queryable state, and an authorization boundary without requiring
external infrastructure.

The model remains a recorded trajectory so the notebook is deterministic and
runs offline.


In [ ]:
from __future__ import annotations

import json
import os
import sqlite3
import time
import uuid
from contextlib import contextmanager
from dataclasses import dataclass
from typing import Any, Callable, Iterator

from pydantic import BaseModel, Field, ValidationError

DB_PATH = "harness_runs.db"
if os.path.exists(DB_PATH):
    os.remove(DB_PATH)

INCIDENT_TICKET = """
At 14:05 UTC, checkout errors increased from below 1% to 18%.
Customers report timeouts after submitting payment.
The checkout service was deployed at 13:52 UTC.
Please identify the likely cause and recommend the safest immediate response.
""".strip()


## 1. Start at the request boundary

A browser or API client can submit the task and later return a `run_id`, but it
must not be trusted to tell the harness which tenant or principal it belongs to.
That identity comes from authenticated server-side context.

The tiny `AuthGateway` below stands in for middleware that has already verified a
session, JWT, mTLS identity, or another credential. The important part is what
the application receives after authentication: a principal the server trusts.


In [ ]:
@dataclass(frozen=True)
class Principal:
    principal_id: str
    tenant_id: str
    roles: frozenset[str]


class AuthGateway:
    # Demo credentials only. Production code receives this identity from
    # verified authentication middleware or an identity provider.
    _tokens = {
        "token-operator-a": Principal("user_42", "tenant_alpha", frozenset({"operator"})),
        "token-commander-a": Principal("user_7", "tenant_alpha", frozenset({"incident_commander"})),
        "token-operator-b": Principal("user_99", "tenant_beta", frozenset({"operator"})),
    }

    def require_principal(self, bearer_token: str) -> Principal:
        try:
            return self._tokens[bearer_token]
        except KeyError:
            raise PermissionError("invalid or expired credential")


@dataclass(frozen=True)
class StartRunRequest:
    task: str


@dataclass(frozen=True)
class ResumeRunRequest:
    run_id: str


auth = AuthGateway()


## 2. Make run identity and workflow state harness-owned

The client does not create the run identifier. The harness creates it after
authentication and stores the tenant and principal alongside the run.

`RunStore.require_access()` is the critical boundary: every read, resume,
refinement, approval, and handoff resolves the public `run_id` through the
authenticated principal. Knowing a run id is therefore not sufficient to access
the task.

The check here is deliberately **tenant-scoped**: any principal in the owning
tenant may read and resume the run, and the narrower question of *which*
principal may act is handled separately by role checks, as section 6 shows for
approval. Per-user ownership, delegation, and separation of duty are policy
decisions that belong on top of this boundary rather than inside it.


In [ ]:
SCHEMA = """
PRAGMA foreign_keys = ON;
CREATE TABLE runs (
    run_id TEXT PRIMARY KEY,
    tenant_id TEXT NOT NULL,
    created_by TEXT NOT NULL,
    task TEXT NOT NULL,
    status TEXT NOT NULL,
    next_step INTEGER NOT NULL,
    max_steps INTEGER NOT NULL,
    -- A named resume point. `next_step` says how far the loop got; `node` says
    -- what the run is waiting on, which is what a resuming process needs first.
    node TEXT NOT NULL,
    model_name TEXT NOT NULL,
    verified_plan_json TEXT,
    created_at REAL NOT NULL,
    updated_at REAL NOT NULL
);
CREATE TABLE messages (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    run_id TEXT NOT NULL REFERENCES runs(run_id),
    role TEXT NOT NULL,
    content TEXT NOT NULL,
    created_at REAL NOT NULL
);
CREATE TABLE events (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    run_id TEXT NOT NULL REFERENCES runs(run_id),
    kind TEXT NOT NULL,
    payload_json TEXT NOT NULL,
    created_at REAL NOT NULL
);
CREATE TABLE pending_actions (
    action_id TEXT PRIMARY KEY,
    run_id TEXT NOT NULL REFERENCES runs(run_id),
    tool TEXT NOT NULL,
    arguments_json TEXT NOT NULL,
    status TEXT NOT NULL,
    requested_at_step INTEGER NOT NULL,
    approved_by TEXT,
    approved_at REAL
);
CREATE TABLE effects (
    effect_id TEXT PRIMARY KEY,
    run_id TEXT NOT NULL REFERENCES runs(run_id),
    action_id TEXT,
    -- UNIQUE is the mutual exclusion. A row is inserted as 'reserved' *before*
    -- the executor is called, so only one caller can ever reach the executor.
    idempotency_key TEXT NOT NULL UNIQUE,
    status TEXT NOT NULL,
    tool TEXT NOT NULL,
    arguments_json TEXT NOT NULL,
    result_json TEXT,
    reserved_at REAL NOT NULL,
    committed_at REAL
);
CREATE TABLE artifacts (
    artifact_id TEXT PRIMARY KEY,
    run_id TEXT NOT NULL REFERENCES runs(run_id),
    name TEXT NOT NULL,
    producer_execution_id TEXT NOT NULL,
    payload_json TEXT NOT NULL,
    created_at REAL NOT NULL
);
"""


class RunStore:
    def __init__(self, path: str = DB_PATH) -> None:
        self.path = path
        with self.connect() as db:
            db.executescript(SCHEMA)

    @contextmanager
    def connect(self) -> Iterator[sqlite3.Connection]:
        """A connection that commits on success and always closes.

        `with sqlite3.connect(...)` commits but does not close, so wrapping it
        here keeps connections from accumulating across the many small
        operations below.
        """
        db = sqlite3.connect(self.path)
        db.row_factory = sqlite3.Row
        db.execute("PRAGMA foreign_keys = ON")
        try:
            with db:
                yield db
        finally:
            db.close()

    def create_run(self, principal: Principal, task: str, model_name: str,
                   max_steps: int = 12) -> str:
        run_id = f"run_{uuid.uuid4().hex}"
        now = time.time()
        with self.connect() as db:
            db.execute(
                """INSERT INTO runs
                   (run_id, tenant_id, created_by, task, status, next_step,
                    max_steps, node, model_name, created_at, updated_at)
                   VALUES (?, ?, ?, ?, 'running', 1, ?, 'gather_evidence', ?, ?, ?)""",
                (run_id, principal.tenant_id, principal.principal_id,
                 task, max_steps, model_name, now, now),
            )
            db.execute(
                "INSERT INTO messages(run_id, role, content, created_at) VALUES (?, 'user', ?, ?)",
                (run_id, task, now),
            )
        return run_id

    def require_access(self, principal: Principal, run_id: str) -> sqlite3.Row:
        with self.connect() as db:
            row = db.execute("SELECT * FROM runs WHERE run_id = ?", (run_id,)).fetchone()
        if row is None:
            raise KeyError(f"unknown run: {run_id}")
        if row["tenant_id"] != principal.tenant_id:
            raise PermissionError("run does not belong to authenticated tenant")
        return row

    def record_event(self, run_id: str, kind: str, payload: dict[str, Any] | None = None) -> None:
        with self.connect() as db:
            db.execute(
                "INSERT INTO events(run_id, kind, payload_json, created_at) VALUES (?, ?, ?, ?)",
                (run_id, kind, json.dumps(payload or {}), time.time()),
            )

    def events(self, run_id: str) -> list[dict[str, Any]]:
        with self.connect() as db:
            rows = db.execute(
                "SELECT kind, payload_json FROM events WHERE run_id = ? ORDER BY id", (run_id,)
            ).fetchall()
        return [{"kind": r["kind"], "payload": json.loads(r["payload_json"])} for r in rows]

    def messages(self, run_id: str) -> list[dict[str, str]]:
        with self.connect() as db:
            rows = db.execute(
                "SELECT role, content FROM messages WHERE run_id = ? ORDER BY id", (run_id,)
            ).fetchall()
        return [dict(r) for r in rows]

    def append_message(self, run_id: str, role: str, content: str) -> None:
        with self.connect() as db:
            db.execute(
                "INSERT INTO messages(run_id, role, content, created_at) VALUES (?, ?, ?, ?)",
                (run_id, role, content, time.time()),
            )

    def checkpoint(self, run_id: str, *, next_step: int | None = None,
                   status: str | None = None, model_name: str | None = None) -> None:
        fields, values = [], []
        if next_step is not None:
            fields.append("next_step = ?"); values.append(next_step)
        if status is not None:
            fields.append("status = ?"); values.append(status)
        if model_name is not None:
            fields.append("model_name = ?"); values.append(model_name)
        fields.append("updated_at = ?"); values.append(time.time())
        values.append(run_id)
        with self.connect() as db:
            db.execute(f"UPDATE runs SET {', '.join(fields)} WHERE run_id = ?", values)

    # -- suspension --------------------------------------------------------
    #
    # `connect()` already gives one transaction per call, which is enough while
    # each write stands alone. Suspending a run does not: the event append and
    # the checkpoint update have to land together or not at all, because an
    # event log that has moved past its checkpoint describes a run nobody can
    # safely resume. The helpers below therefore take an open connection so a
    # caller can put several writes inside one transaction.

    transaction = connect

    def append_events(self, db: sqlite3.Connection, run_id: str,
                      events: list[dict[str, Any]]) -> None:
        now = time.time()
        db.executemany(
            "INSERT INTO events(run_id, kind, payload_json, created_at) VALUES (?, ?, ?, ?)",
            [(run_id, e["kind"], json.dumps(e.get("payload", {})), now) for e in events],
        )

    def write_checkpoint(self, db: sqlite3.Connection, run_id: str, *,
                         node: str, status: str, next_step: int | None = None) -> None:
        if next_step is None:
            db.execute(
                "UPDATE runs SET node = ?, status = ?, updated_at = ? WHERE run_id = ?",
                (node, status, time.time(), run_id),
            )
        else:
            db.execute(
                """UPDATE runs SET node = ?, status = ?, next_step = ?, updated_at = ?
                   WHERE run_id = ?""",
                (node, status, next_step, time.time(), run_id),
            )

    def load_checkpoint(self, principal: Principal, run_id: str) -> dict[str, Any]:
        row = self.require_access(principal, run_id)
        return {"run_id": row["run_id"], "task": row["task"], "node": row["node"],
                "status": row["status"], "next_step": row["next_step"],
                "model_name": row["model_name"]}

    def save_verified_plan(self, run_id: str, plan: dict[str, Any]) -> None:
        with self.connect() as db:
            db.execute(
                """UPDATE runs SET verified_plan_json = ?, status = 'completed', updated_at = ?
                   WHERE run_id = ?""",
                (json.dumps(plan), time.time(), run_id),
            )

    def verified_plan(self, run_id: str) -> dict[str, Any] | None:
        with self.connect() as db:
            row = db.execute(
                "SELECT verified_plan_json FROM runs WHERE run_id = ?", (run_id,)
            ).fetchone()
        return json.loads(row["verified_plan_json"]) if row and row["verified_plan_json"] else None

    def request_action(self, run_id: str, tool: str, arguments: dict[str, Any], step: int) -> str:
        action_id = f"act_{uuid.uuid4().hex}"
        with self.connect() as db:
            db.execute(
                """INSERT INTO pending_actions
                   (action_id, run_id, tool, arguments_json, status, requested_at_step)
                   VALUES (?, ?, ?, ?, 'pending', ?)""",
                (action_id, run_id, tool, json.dumps(arguments, sort_keys=True), step),
            )
        return action_id

    def pending_action(self, run_id: str, action_id: str) -> sqlite3.Row:
        with self.connect() as db:
            row = db.execute(
                "SELECT * FROM pending_actions WHERE run_id = ? AND action_id = ?",
                (run_id, action_id),
            ).fetchone()
        if row is None:
            raise KeyError("pending action not found for this run")
        return row

    def latest_pending_action(self, run_id: str) -> sqlite3.Row | None:
        with self.connect() as db:
            return db.execute(
                """SELECT * FROM pending_actions WHERE run_id = ? AND status = 'pending'
                   ORDER BY rowid DESC LIMIT 1""",
                (run_id,),
            ).fetchone()

    def approve_action(self, run_id: str, action_id: str, principal: Principal) -> None:
        if "incident_commander" not in principal.roles:
            raise PermissionError("incident_commander role required")
        with self.connect() as db:
            cur = db.execute(
                """UPDATE pending_actions
                   SET status = 'approved', approved_by = ?, approved_at = ?
                   WHERE run_id = ? AND action_id = ? AND status = 'pending'""",
                (principal.principal_id, time.time(), run_id, action_id),
            )
            if cur.rowcount != 1:
                raise RuntimeError("action is not pending")

    def effect(self, idempotency_key: str) -> sqlite3.Row | None:
        with self.connect() as db:
            return db.execute(
                "SELECT * FROM effects WHERE idempotency_key = ?", (idempotency_key,)
            ).fetchone()

    def reserve_effect(self, run_id: str, action_id: str | None, idempotency_key: str,
                       tool: str, arguments: dict[str, Any]) -> None:
        """Claim the right to execute. Raises IntegrityError if already claimed.

        This is the whole mechanism. Checking whether an effect exists and *then*
        calling the executor leaves a window in which two callers both pass the
        check and production changes twice; the UNIQUE constraint would then
        protect the record but not the world. Inserting first closes the window,
        because only one caller can win the insert.
        """
        with self.connect() as db:
            db.execute(
                """INSERT INTO effects
                   (effect_id, run_id, action_id, idempotency_key, status, tool,
                    arguments_json, reserved_at)
                   VALUES (?, ?, ?, ?, 'reserved', ?, ?, ?)""",
                (f"eff_{uuid.uuid4().hex}", run_id, action_id, idempotency_key, tool,
                 json.dumps(arguments, sort_keys=True), time.time()),
            )

    def commit_effect(self, idempotency_key: str, action_id: str | None,
                      result: dict[str, Any]) -> None:
        with self.connect() as db:
            db.execute(
                """UPDATE effects SET status = 'committed', result_json = ?,
                   committed_at = ? WHERE idempotency_key = ?""",
                (json.dumps(result), time.time(), idempotency_key),
            )
            if action_id:
                db.execute("UPDATE pending_actions SET status = 'executed' WHERE action_id = ?", (action_id,))

    def release_reservation(self, idempotency_key: str) -> None:
        """Undo a claim whose executor failed, so the action can be retried."""
        with self.connect() as db:
            db.execute(
                "DELETE FROM effects WHERE idempotency_key = ? AND status = 'reserved'",
                (idempotency_key,),
            )

    def write_artifact(self, run_id: str, name: str, producer_execution_id: str,
                       payload: dict[str, Any]) -> str:
        artifact_id = f"art_{uuid.uuid4().hex}"
        with self.connect() as db:
            db.execute(
                """INSERT INTO artifacts
                   (artifact_id, run_id, name, producer_execution_id, payload_json, created_at)
                   VALUES (?, ?, ?, ?, ?, ?)""",
                (artifact_id, run_id, name, producer_execution_id, json.dumps(payload), time.time()),
            )
            db.execute(
                "INSERT INTO events(run_id, kind, payload_json, created_at) VALUES (?, ?, ?, ?)",
                (run_id, "artifact_written",
                 json.dumps({"name": name, "artifact_id": artifact_id,
                             "producer_execution_id": producer_execution_id,
                             "payload": payload}), time.time()),
            )
        return artifact_id

    def latest_artifact(self, run_id: str, name: str) -> dict[str, Any] | None:
        with self.connect() as db:
            row = db.execute(
                """SELECT * FROM artifacts WHERE run_id = ? AND name = ?
                   ORDER BY created_at DESC LIMIT 1""",
                (run_id, name),
            ).fetchone()
        if row is None:
            return None
        return {
            "artifact_id": row["artifact_id"],
            "producer_execution_id": row["producer_execution_id"],
            "payload": json.loads(row["payload_json"]),
        }


store = RunStore()


### Create the run from the UI request

Notice what the client does **not** send: no trusted `tenant_id`, no trusted
`principal_id`, no workflow position, and no prior conversation state.


In [ ]:
def start_run(request: StartRunRequest, bearer_token: str) -> str:
    principal = auth.require_principal(bearer_token)
    return store.create_run(principal, request.task, model_name="replay-model")


run_id = start_run(StartRunRequest(task=INCIDENT_TICKET), "token-operator-a")
print("client receives:", run_id)

principal_a = auth.require_principal("token-operator-a")
row = store.require_access(principal_a, run_id)
print("server resolved :", row["tenant_id"], row["created_by"], row["status"])


client receives: run_e7c376068d4945aeb04fb5e8c21aa01c
server resolved : tenant_alpha user_42 running


### A run id is a locator, not an authorization credential

The next request knows the correct UUID but comes from another authenticated
tenant. The store rejects it before any workflow state is returned.


In [ ]:
principal_b = auth.require_principal("token-operator-b")
try:
    store.require_access(principal_b, run_id)
except PermissionError as exc:
    print(type(exc).__name__ + ":", exc)


PermissionError: run does not belong to authenticated tenant


## 3. Operational state and governed effects

The incident environment is the same as in Part 1. The difference is that a
side-effecting tool is now executed only through a pending action with its own
identity. The action can survive process boundaries and later receive an
authenticated approval.


In [ ]:
def fresh_environment() -> dict[str, Any]:
    return {
        "service_status": {"checkout": {
            "error_rate_pct": 18.2, "db_pool_in_use": 20, "db_pool_size": 20,
            "release": "checkout-2026.07.29.3",
        }},
        "logs": {"checkout": [
            "14:04:58 ERROR acquire connection timeout after 5000ms",
            "14:05:01 WARN db pool saturated: 20/20 connections in use",
        ]},
        "deployments": {"checkout-2026.07.29.3": {
            "deployed_at": "13:52 UTC",
            "changes": {"DB_POOL_SIZE": {"from": 80, "to": 20}},
            "rollback_available": True,
            "previous_release": "checkout-2026.07.24.1",
        }},
    }


env = fresh_environment()


def build_registry(env: dict[str, Any]) -> dict[str, Callable[..., dict[str, Any]]]:
    def get_service_status(service: str) -> dict[str, Any]:
        return env["service_status"].get(service, {"error": f"Unknown: {service}"})

    def search_logs(service: str, query: str) -> dict[str, Any]:
        lines = env["logs"].get(service, [])
        return {"service": service, "matches": [line for line in lines if query.lower() in line.lower()]}

    def get_deployment(release: str) -> dict[str, Any]:
        if release == "checkout":
            release = env["service_status"]["checkout"]["release"]
        return env["deployments"].get(release, {"error": f"Unknown: {release}"})

    def rollback_service(service: str, *, idempotency_key: str) -> dict[str, Any]:
        status = env["service_status"][service]
        current = status["release"]
        previous = env["deployments"][current]["previous_release"]
        status["release"] = previous
        status["error_rate_pct"] = 0.4
        status["db_pool_size"] = 80
        return {"service": service, "rolled_back_from": current,
                "now_running": previous, "idempotency_key": idempotency_key}

    return {"get_service_status": get_service_status, "search_logs": search_logs,
            "get_deployment": get_deployment, "rollback_service": rollback_service}


registry = build_registry(env)
SIDE_EFFECTING = {"rollback_service"}


class IncidentPlan(BaseModel):
    likely_cause: str
    confidence: str
    immediate_actions: list[str] = Field(min_length=1)
    evidence: list[str] = Field(min_length=1)
    escalation_condition: str


## 4. Reconstruct model context from durable run state

The model sees a projection. It does not own the workflow record.

The recorded model trajectory keeps the example deterministic, but each model
call is selected by the checkpointed `next_step`, so a new Python process can
construct a new client and continue at the correct step.


In [ ]:
@dataclass
class _Function:
    name: str
    arguments: str

@dataclass
class _ToolCall:
    id: str
    function: _Function

@dataclass
class _Message:
    content: str | None = None
    tool_calls: list[_ToolCall] | None = None

@dataclass
class _Choice:
    message: _Message
    finish_reason: str

@dataclass
class _Response:
    choices: list[_Choice]


_GOOD_PLAN = {
    "likely_cause": "The checkout deployment at 13:52 UTC reduced DB_POOL_SIZE from 80 to 20, exhausting the database connection pool.",
    "confidence": "Very high",
    "immediate_actions": ["Request approval to roll back checkout and verify recovery."],
    "evidence": [
        "Source get_service_status: 20 of 20 database connections are in use.",
        "Source get_deployment: DB_POOL_SIZE changed from 80 to 20.",
        "Source incident ticket: checkout errors rose to 18% at 14:05 UTC.",
    ],
    "escalation_condition": "Escalate if rollback cannot be approved.",
}
_MALFORMED_PLAN = {**_GOOD_PLAN, "confidence": 0.99}
RECORDED_RUN = [
    {"tool_calls": [
        {"name": "get_service_status", "arguments": {"service": "checkout"}},
        {"name": "search_logs", "arguments": {"service": "checkout", "query": "timeout"}},
        {"name": "get_deployment", "arguments": {"release": "checkout"}},
    ]},
    {"tool_calls": [{"name": "get_deployment", "arguments": {"release": "checkout-2026.07.29.3"}}]},
    {"tool_calls": [{"name": "rollback_service", "arguments": {"service": "checkout"}}]},
    {"content": json.dumps(_MALFORMED_PLAN)},
    {"content": json.dumps(_GOOD_PLAN)},
]


class ReplayExhausted(RuntimeError):
    """The recording ran out before the harness stopped asking."""


class ReplayClient:
    def __init__(self, turns: list[dict[str, Any]]) -> None:
        self.turns = turns

    def create_for_step(self, step: int, **_: Any) -> _Response:
        turn = self.turns[step - 1]
        if "tool_calls" in turn:
            calls = [_ToolCall(f"call_{step}_{i}", _Function(tc["name"], json.dumps(tc["arguments"])))
                     for i, tc in enumerate(turn["tool_calls"])]
            return _Response([_Choice(_Message(tool_calls=calls), "tool_calls")])
        return _Response([_Choice(_Message(content=turn["content"]), "stop")])


In [ ]:
class ToolGateway:
    def __init__(self, store: RunStore, registry: dict[str, Callable[..., dict[str, Any]]]) -> None:
        self.store = store
        self.registry = registry

    def invoke(self, run_id: str, step: int, tool: str, arguments: dict[str, Any]) -> dict[str, Any]:
        if tool in SIDE_EFFECTING:
            action_id = self.store.request_action(run_id, tool, arguments, step)
            self.store.record_event(run_id, "approval_requested",
                                    {"action_id": action_id, "tool": tool, "step": step})
            return {"status": "approval_required", "action_id": action_id, "tool": tool}
        self.store.record_event(run_id, "tool_started", {"tool": tool, "step": step})
        result = self.registry[tool](**arguments)
        self.store.record_event(run_id, "tool_completed",
                                {"tool": tool, "step": step, "result": result})
        return result


@dataclass
class RunResult:
    run_id: str
    plan: IncidentPlan | None
    status: str


class Orchestrator:
    def __init__(self, store: RunStore, gateway: ToolGateway, client: ReplayClient) -> None:
        self.store = store; self.gateway = gateway; self.client = client

    def run(self, principal: Principal, run_id: str,
            *, stop_before_step: int | None = None) -> RunResult:
        run = self.store.require_access(principal, run_id)
        verified = self.store.verified_plan(run_id)
        if verified is not None:
            return RunResult(run_id, IncidentPlan.model_validate(verified), "completed")
        step = run["next_step"]
        # The budget is a stored policy, not the length of the recording.
        while step <= run["max_steps"]:
            if stop_before_step == step:
                raise RuntimeError(f"simulated process crash before step {step}")
            self.store.record_event(run_id, "model_call_started", {"step": step})
            response = self.client.create_for_step(step, messages=self.store.messages(run_id))
            choice = response.choices[0]
            self.store.record_event(run_id, "model_call_completed",
                                    {"step": step, "finish_reason": choice.finish_reason})
            if choice.finish_reason == "tool_calls":
                for call in choice.message.tool_calls or []:
                    arguments = json.loads(call.function.arguments)
                    result = self.gateway.invoke(run_id, step, call.function.name, arguments)
                    self.store.append_message(run_id, "tool",
                                              json.dumps({"tool": call.function.name, "result": result}))
                step += 1
                self.store.checkpoint(run_id, next_step=step)
                continue
            try:
                plan = IncidentPlan.model_validate_json(choice.message.content or "")
                # Grounding reads the event history, so the set of sources the
                # plan may cite survives a process restart along with everything
                # else. Part 1 removes this edge and shows what gets accepted.
                observed = {e["payload"]["tool"] for e in self.store.events(run_id)
                            if e["kind"] == "tool_completed"} | {"incident ticket"}
                ungrounded = [item for item in plan.evidence
                              if not any(source in item for source in observed)]
                if ungrounded:
                    raise ValueError(
                        f"evidence cites a source that produced no observation "
                        f"on this run: {ungrounded}")
            except (ValidationError, ValueError) as exc:
                self.store.record_event(run_id, "verification_failed", {"step": step, "error": str(exc)})
                self.store.append_message(run_id, "user",
                                          "Verification failed. Return a valid IncidentPlan JSON object.")
                step += 1
                self.store.checkpoint(run_id, next_step=step)
                continue
            self.store.record_event(run_id, "verification_passed", {"step": step})
            self.store.save_verified_plan(run_id, plan.model_dump())
            return RunResult(run_id, plan, "completed")
        self.store.checkpoint(run_id, status="escalated")
        return RunResult(run_id, None, "escalated")


gateway = ToolGateway(store, registry)


## 5. Cross a process boundary

The first process runs step 1, checkpoints `next_step = 2`, and then dies before
step 2. Nothing in the second process needs the first process's Python objects.
It authenticates the caller, resolves the run, and reconstructs the next model
context from the durable store.


In [ ]:
process_a = Orchestrator(store, gateway, ReplayClient(RECORDED_RUN))
try:
    process_a.run(principal_a, run_id, stop_before_step=2)
except RuntimeError as exc:
    print(exc)
checkpoint = store.require_access(principal_a, run_id)
print("checkpointed next_step:", checkpoint["next_step"])
print("messages persisted     :", len(store.messages(run_id)))


simulated process crash before step 2
checkpointed next_step: 2
messages persisted     : 4


In [ ]:
process_b = Orchestrator(store, gateway, ReplayClient(RECORDED_RUN))
result = process_b.run(principal_a, run_id)
print("status      :", result.status)
print("confidence  :", result.plan.confidence)
print("next_step   :", store.require_access(principal_a, run_id)["next_step"])
pending = store.latest_pending_action(run_id)
grounded_on = sorted({e["payload"]["tool"] for e in store.events(run_id)
                      if e["kind"] == "tool_completed"})
print("evidence could cite:", grounded_on)
print("pending action:", pending["action_id"], pending["tool"], pending["status"])


status      : completed
confidence  : Very high
next_step   : 5
evidence could cite: ['get_deployment', 'get_service_status', 'search_logs']
pending action: act_96b4b0f83605473e9e0c08e14a698ad9 rollback_service pending


The task crossed the process boundary because the harness persisted the workflow
position, messages needed to reconstruct context, pending action, events, and
verified result. The model call itself remained temporary.

## 6. Cross a human-time boundary with an authenticated approval

The pending rollback is not represented by the string `"rollback_service"`.
It has a durable `action_id`, arguments, run ownership, and status.

A later request may carry the `run_id` and `action_id`, but the approver identity
again comes from authentication. The store first checks that the run belongs to
the authenticated tenant, and the policy then requires the
`incident_commander` role.


In [ ]:
def approve_action(run_id: str, action_id: str, bearer_token: str) -> None:
    principal = auth.require_principal(bearer_token)
    store.require_access(principal, run_id)
    store.pending_action(run_id, action_id)
    store.approve_action(run_id, action_id, principal)
    with store.transaction() as db:
        store.write_checkpoint(db, run_id, node="release_action",
                               status="ready_to_resume")
    store.record_event(run_id, "approval_granted",
                       {"action_id": action_id, "approved_by": principal.principal_id})


try:
    approve_action(run_id, pending["action_id"], "token-operator-a")
except PermissionError as exc:
    print("operator:", type(exc).__name__ + ":", exc)
approve_action(run_id, pending["action_id"], "token-commander-a")
approved = store.pending_action(run_id, pending["action_id"])
print("commander: approved by", approved["approved_by"])


operator: PermissionError: incident_commander role required
commander: approved by user_7



## 7. Commit the effect exactly once

Approval and execution are separate transitions. When the approved action is
released, the harness creates a durable idempotency key and passes it to the
downstream executor.

Note the order of operations, because it is the whole mechanism. The obvious
shape — look up whether the effect exists, and call the executor if it does not
— leaves a window between the check and the record in which two callers both
find nothing and both call the executor. The `UNIQUE` constraint would then
reject the second *record* after the second *rollback* had already happened,
which is the failure this section exists to prevent.

So the key is **reserved before the executor is called**. Whoever wins the
insert is the only caller that can reach the executor; everyone else either
receives the committed result or is told the action is in flight.

The local function below still stands in for a real deployment API, but the
interface now carries the property that matters in production: the same key
passed twice must not produce a second effect. The harness enforces that on its
side, and a remote service must enforce it on its own.


In [ ]:
def execute_approved_action(run_id: str, action_id: str,
                            bearer_token: str) -> dict[str, Any]:
    principal = auth.require_principal(bearer_token)
    store.require_access(principal, run_id)
    action = store.pending_action(run_id, action_id)
    if action["status"] not in {"approved", "executed"}:
        raise PermissionError("action has not been approved")

    idempotency_key = f"{run_id}:{action_id}"
    arguments = json.loads(action["arguments_json"])

    try:
        store.reserve_effect(run_id, action_id, idempotency_key,
                             action["tool"], arguments)
    except sqlite3.IntegrityError:
        # Someone already claimed this key. Either they finished, or they are
        # still running and this caller must not call the executor as well.
        prior = store.effect(idempotency_key)
        if prior["status"] == "committed":
            store.record_event(run_id, "effect_replayed",
                               {"action_id": action_id, "idempotency_key": idempotency_key})
            return json.loads(prior["result_json"])
        raise RuntimeError("this action is already in flight in another process")

    try:
        result = registry[action["tool"]](**arguments, idempotency_key=idempotency_key)
    except Exception:
        # The executor failed, so release the claim and let the action be retried.
        # A crash between reservation and commit leaves the row 'reserved'; real
        # systems reconcile those with a lease timeout rather than a try block.
        store.release_reservation(idempotency_key)
        raise

    store.commit_effect(idempotency_key, action_id, result)
    store.record_event(run_id, "effect_committed",
                       {"action_id": action_id, "idempotency_key": idempotency_key})
    return result


before = env["service_status"]["checkout"]["release"]
effect = execute_approved_action(run_id, pending["action_id"], "token-commander-a")
after = env["service_status"]["checkout"]["release"]
print("release:", before, "->", after)
print("idempotency key:", effect["idempotency_key"])


release: checkout-2026.07.29.3 -> checkout-2026.07.24.1
idempotency key: run_e7c376068d4945aeb04fb5e8c21aa01c:act_96b4b0f83605473e9e0c08e14a698ad9


In [ ]:
again = execute_approved_action(run_id, pending["action_id"], "token-commander-a")
print("same stored result :", again == effect)
print("current release    :", env["service_status"]["checkout"]["release"])
effect_events = [e["kind"] for e in store.events(run_id)
                 if e["kind"] in {"effect_committed", "effect_replayed"}]
print("effect events      :", effect_events)


same stored result : True
current release    : checkout-2026.07.24.1
effect events      : ['effect_committed', 'effect_replayed']


### Two callers at once

The replay above shows a *sequential* second call. The harder case is two
callers arriving together — a retried webhook, two workers polling the same
queue, an operator clicking twice.

Reserving the key is what makes that safe, so it is worth watching the
mechanism directly rather than trusting the description. Both threads below
attempt the same fresh key.

In [ ]:
import threading

DEMO_KEY = f"{run_id}:demo-concurrency"
executor_calls: list[str] = []


def attempt(label: str, results: dict[str, str]) -> None:
    try:
        store.reserve_effect(run_id, None, DEMO_KEY, "rollback_service",
                             {"service": "checkout"})
    except sqlite3.IntegrityError:
        results[label] = "refused: another caller holds the key"
        return
    time.sleep(0.05)                      # the window a check-then-act would lose in
    executor_calls.append(label)          # only a reservation holder gets here
    store.commit_effect(DEMO_KEY, None, {"ok": True})
    results[label] = "executed"


results: dict[str, str] = {}
threads = [threading.Thread(target=attempt, args=(name, results))
           for name in ("caller-1", "caller-2")]
for t in threads:
    t.start()
for t in threads:
    t.join()

print("outcomes       :", results)
print("executor ran   :", len(executor_calls), "time(s)")
print("effect rows    :", 1 if store.effect(DEMO_KEY) else 0)

outcomes       : {'caller-2': 'refused: another caller holds the key', 'caller-1': 'executed'}
executor ran   : 1 time(s)
effect rows    : 1


One reservation, one execution. Had the harness checked for an existing effect
and *then* called the executor, both threads would have found nothing, both
would have rolled the service back, and the `UNIQUE` constraint would have
rejected only the second bookkeeping row — after the second rollback had already
happened.

That is the difference between protecting the record and protecting the world.

## 8. Cross a client boundary

The browser is a view over harness state, not the owner of it. A later request
therefore needs only the authenticated credential, the `run_id`, and the new
instruction. It does not resend trusted tenant identity, workflow position,
approvals, effects, or the old transcript.

The server resolves all of that state before it accepts the refinement.


In [ ]:
def request_refinement(run_id: str, instruction: str, bearer_token: str) -> None:
    principal = auth.require_principal(bearer_token)
    store.require_access(principal, run_id)
    store.append_message(run_id, "user", instruction)
    store.record_event(run_id, "refinement_requested",
                       {"instruction": instruction, "requested_by": principal.principal_id})
    with store.connect() as db:
        db.execute(
            """UPDATE runs SET verified_plan_json = NULL, status = 'running', updated_at = ?
               WHERE run_id = ?""",
            (time.time(), run_id),
        )


request_refinement(run_id, "Tighten the escalation condition to a numeric threshold.",
                   "token-operator-a")
print("client sent       : authenticated request + run_id + instruction")
print("server has tenant :", store.require_access(principal_a, run_id)["tenant_id"])
print("stored messages   :", len(store.messages(run_id)))


client sent       : authenticated request + run_id + instruction
server has tenant : tenant_alpha
stored messages   : 8


For the refinement itself, use a fresh model step rather than replaying the
incident trajectory. The important part is that the prior task state remains
server-side and the new context is constructed from it.


In [ ]:
REFINED_PLAN = {
    "likely_cause": "DB_POOL_SIZE was reduced from 80 to 20 by the 13:52 deployment.",
    "confidence": "high",
    "immediate_actions": ["Rollback executed; verify pool utilisation recovered."],
    "evidence": [
        "Source get_deployment: DB_POOL_SIZE changed from 80 to 20.",
        "Source incident ticket: errors rose to 18% at 14:05 UTC.",
    ],
    "escalation_condition": "Escalate if the checkout error rate exceeds 2% for 5 consecutive minutes.",
}
store.save_verified_plan(run_id, REFINED_PLAN)
print(store.verified_plan(run_id)["escalation_condition"])


Escalate if the checkout error rate exceeds 2% for 5 consecutive minutes.


## 9. Cross a model boundary

The task identity is not the model identity. A later step can use another model
because the harness owns the run, the workflow position, and the artifacts from
earlier execution.


In [ ]:
store.checkpoint(run_id, model_name="different-model")
run = store.require_access(principal_a, run_id)
print("run_id     :", run["run_id"])
print("model_name :", run["model_name"])
print("plan exists:", store.verified_plan(run_id) is not None)


run_id     : run_e7c376068d4945aeb04fb5e8c21aa01c
model_name : different-model
plan exists: True


## 10. Externalize the run: suspend and rehydrate

The boundaries above were each crossed with a specific mechanism. Underneath all
of them is one pair of operations: freeze an active execution into durable
state, and reconstruct it later without the process that created it.

Suspension is where atomicity starts to matter. Two writes have to happen —
append what was observed, and move the checkpoint — and they have to land
together. A log that has advanced past its checkpoint describes a run that
cannot be safely resumed, because replaying from the checkpoint would repeat
work the log says already happened.

In [ ]:
class StateError(RuntimeError):
    """The run is not in a state this transition allows."""


def suspend_run(run_id: str, node: str, state_delta: list[dict[str, Any]],
                status: str = "waiting_for_approval") -> None:
    """Freeze active execution into durable state, atomically."""
    with store.transaction() as db:
        store.append_events(db, run_id, state_delta)
        store.write_checkpoint(db, run_id, node=node, status=status)


run_b = start_run(StartRunRequest(task=INCIDENT_TICKET), "token-operator-a")
process = Orchestrator(store, gateway, ReplayClient(RECORDED_RUN))
try:
    process.run(principal_a, run_b, stop_before_step=4)
except RuntimeError as exc:
    print(exc)

suspend_run(run_b, node="await_approval", state_delta=[
    {"kind": "suspended", "payload": {"reason": "waiting for incident commander"}},
])

checkpoint = store.load_checkpoint(principal_a, run_b)
print("node   :", checkpoint["node"])
print("status :", checkpoint["status"])
print("events :", len(store.events(run_b)))

simulated process crash before step 4
node   : await_approval
status : waiting_for_approval
events : 16


### The transaction is not decoration

Forcing the second write to fail shows what the transaction is for. Neither the
appended events nor the checkpoint change survive.

In [ ]:
before_events = len(store.events(run_b))
before_node = store.load_checkpoint(principal_a, run_b)["node"]

def suspend_with_failure(run_id: str) -> None:
    with store.transaction() as db:
        store.append_events(db, run_id, [{"kind": "suspended", "payload": {}}])
        store.write_checkpoint(db, run_id, node=None, status="broken")  # NOT NULL

try:
    suspend_with_failure(run_b)
except sqlite3.IntegrityError as exc:
    print("suspend failed:", exc)

print("events after :", len(store.events(run_b)), "(was", before_events, ")")
print("node after   :", store.load_checkpoint(principal_a, run_b)["node"],
      "(was", before_node + ")")

suspend failed: NOT NULL constraint failed: runs.node
events after : 16 (was 16 )
node after   : await_approval (was await_approval)


### Rehydration folds the log

The model remembers nothing, so the harness reconstructs the task before the
model is invoked again. The reconstruction reads the event history and folds it
into workflow state — the model's own account of what happened is not consulted,
because the log records what the harness observed rather than what the model
claimed.

In [ ]:
def build_state_from_events(principal: Principal, run_id: str) -> dict[str, Any]:
    """Fold the immutable history into the workflow state a step needs."""
    checkpoint = store.load_checkpoint(principal, run_id)
    state: dict[str, Any] = {
        **checkpoint,
        "observed_tools": set(),
        "pending_action": None,
        "approved_actions": set(),
        "committed_effects": {},
        "artifacts": {},
        "verification_failures": 0,
    }

    for event in store.events(run_id):
        kind, payload = event["kind"], event["payload"]
        if kind == "tool_completed":
            state["observed_tools"].add(payload["tool"])
        elif kind == "approval_requested":
            state["pending_action"] = payload
        elif kind == "approval_granted":
            state["approved_actions"].add(payload["action_id"])
        elif kind == "effect_committed":
            state["committed_effects"][payload["idempotency_key"]] = payload
        elif kind == "artifact_written":
            state["artifacts"][payload["name"]] = payload["payload"]
        elif kind == "verification_failed":
            state["verification_failures"] += 1

    return state


def project_for_agent(state: dict[str, Any], agent_role: str) -> dict[str, Any]:
    """Scope reconstructed state down to what one agent needs to see."""
    if agent_role == "lead":
        return {"goal": state["task"],
                "deployment_evidence": state["artifacts"].get("deployment_evidence")}
    if agent_role == "deployment_subagent":
        return {"task": "Inspect the deployment that preceded the checkout incident.",
                "release": "checkout-2026.07.29.3"}
    if agent_role == "release_action":
        return {"goal": state["task"], "pending_action": state["pending_action"]}
    raise KeyError(agent_role)


state_b = build_state_from_events(principal_a, run_b)
print("node               :", state_b["node"])
print("observed tools     :", sorted(state_b["observed_tools"]))
print("verification fails :", state_b["verification_failures"])
print("pending action     :", (state_b["pending_action"] or {}).get("tool"))

node               : await_approval
observed tools     : ['get_deployment', 'get_service_status', 'search_logs']
verification fails : 0
pending action     : rollback_service


Two properties of that fold are worth naming. It is derived rather than stored,
so it cannot drift from the history it summarises. And it is recomputable by any
process holding the run id and the right credential, which is what makes the
next cell possible at all.

In [ ]:
def resume_run(run_id: str, bearer_token: str) -> dict[str, Any]:
    principal = auth.require_principal(bearer_token)                       # 1
    checkpoint = store.load_checkpoint(principal, run_id)
    if checkpoint["status"] != "ready_to_resume":
        raise StateError(f"cannot resume a run in status: {checkpoint['status']}")

    workflow_state = build_state_from_events(principal, run_id)            # 2
    model_context = project_for_agent(workflow_state, checkpoint["node"])  # 3
    store.record_event(run_id, "run_resumed", {"node": checkpoint["node"]})
    return model_context


try:
    resume_run(run_b, "token-operator-a")
except StateError as exc:
    print("StateError:", exc)

with store.transaction() as db:
    store.write_checkpoint(db, run_b, node="release_action", status="ready_to_resume")

context = resume_run(run_b, "token-operator-a")
print("rehydrated context :", sorted(context))
print("pending action     :", (context["pending_action"] or {}).get("tool"))

StateError: cannot resume a run in status: waiting_for_approval
rehydrated context : ['goal', 'pending_action']
pending action     : rollback_service


Rehydration begins at the authorization boundary (1), reconstructs workflow state
by folding the history (2), and only then projects a scoped context for the next
inference (3). A run in the wrong status is refused before any of that happens.

What crosses the boundary is data. What resumes is the task, not the process.

## 11. Cross an agent boundary without shared model memory

A lead agent and subagent operate inside the same authorized run but receive
different projections. The subagent writes an explicit artifact back to
harness-owned state, including provenance about which execution produced it.

The lead's next context is reconstructed from that artifact. No model transcript
is copied from one agent into another.


In [ ]:
lead_before = project_for_agent(
    build_state_from_events(principal_a, run_id), "lead")
subagent_view = project_for_agent(
    build_state_from_events(principal_a, run_id), "deployment_subagent")
execution_id = f"exec_{uuid.uuid4().hex}"
artifact_id = store.write_artifact(
    run_id, "deployment_evidence", execution_id,
    {"release": subagent_view["release"],
     "change": {"DB_POOL_SIZE": {"from": 80, "to": 20}},
     "assessment": "deployment reduced the connection pool before errors increased"},
)
lead_after = project_for_agent(
    build_state_from_events(principal_a, run_id), "lead")
print("lead before:", lead_before["deployment_evidence"])
print("artifact id :", artifact_id)
print("lead after  :", lead_after["deployment_evidence"])


lead before: None
artifact id : art_0cd607a9de804b5c893c01d498dfa81d
lead after  : {'release': 'checkout-2026.07.29.3', 'change': {'DB_POOL_SIZE': {'from': 80, 'to': 20}}, 'assessment': 'deployment reduced the connection pool before errors increased'}


## 12. What the harness owns

The examples above leave the client and model intentionally thin. The durable
run boundary owns the information required to continue safely:

| Harness-owned state | Why it must survive |
|---|---|
| `run_id` + tenant scope | Locates the task without turning the identifier into authorization |
| authenticated principal on each request | Prevents the client from asserting ownership |
| workflow checkpoint | Reconstructs where execution continues after process loss |
| model-context source data | Lets the harness build the next model projection |
| pending action identity | Makes approval refer to one concrete proposed effect |
| approval record | Preserves who approved what and under which run |
| idempotency key + effect record | Prevents retries or duplicate delivery from repeating an external effect |
| verified artifacts | Preserves results independently of the model call that produced them |
| agent artifacts + provenance | Moves state across agent boundaries without shared model memory |
| event history | Makes the transitions inspectable and auditable |

The storage implementation can change. SQLite may become Postgres, a workflow
engine, or another transactional checkpoint store. The architectural boundary
should not: **the model proposes over a projection, the client refers to the run,
and the harness owns and authorizes the durable state that lets execution
continue.**



## 13. Production extensions beyond this notebook

The ownership, authorization, and exactly-once boundaries are modelled directly
above. What remains is infrastructure that later chapters take on:

- replace the demo token map with real authentication middleware or an identity
  provider;
- use database row-level security or equivalent tenant isolation, rather than a
  comparison in application code;
- reconcile abandoned reservations with a lease timeout, since a process that
  dies between reserving and committing leaves a row no `try` block will clean up;
- pass idempotency keys through to remote services that can honour them, because
  a key only the harness respects protects one side of the boundary;
- add expiry, revocation, and separation-of-duty rules to approvals;
- bound total attempts across resumes, not only the per-run `max_steps` above;
- define retention, redaction, and encryption policies for stored model context;
- compact long-running model context while retaining the full trajectory for audit;
- run tools in sandboxed, timeout-bounded execution rather than in-process.

Those are implementation choices around the boundary demonstrated here, not a
replacement for it.


In [ ]:
for event in store.events(run_id):
    if event["kind"] in {"approval_requested", "verification_failed",
                         "verification_passed", "approval_granted",
                         "effect_committed", "effect_replayed",
                         "refinement_requested"}:
        print(f"{event['kind']:24} {event['payload']}")


approval_requested       {'action_id': 'act_96b4b0f83605473e9e0c08e14a698ad9', 'tool': 'rollback_service', 'step': 3}
verification_failed      {'step': 4, 'error': '1 validation error for IncidentPlan\nconfidence\n  Input should be a valid string [type=string_type, input_value=0.99, input_type=float]\n    For further information visit https://errors.pydantic.dev/2.13/v/string_type'}
verification_passed      {'step': 5}
approval_granted         {'action_id': 'act_96b4b0f83605473e9e0c08e14a698ad9', 'approved_by': 'user_7'}
effect_committed         {'action_id': 'act_96b4b0f83605473e9e0c08e14a698ad9', 'idempotency_key': 'run_e7c376068d4945aeb04fb5e8c21aa01c:act_96b4b0f83605473e9e0c08e14a698ad9'}
effect_replayed          {'action_id': 'act_96b4b0f83605473e9e0c08e14a698ad9', 'idempotency_key': 'run_e7c376068d4945aeb04fb5e8c21aa01c:act_96b4b0f83605473e9e0c08e14a698ad9'}
refinement_requested     {'instruction': 'Tighten the escalation condition to a numeric threshold.', 'requested_by': 'user_